In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# טרנספילציה של מעגלים מרחוק עם שירות Qiskit Transpiler

> **Danger:** החל מ-18 ביולי 2025, השירות עובר העברה לתמיכה בפלטפורמת IBM Quantum&reg; החדשה ואינו זמין. עבור פאסי AI, תוכל להשתמש ב[מצב מקומי](/guides/ai-transpiler-passes#run-the-ai-transpiler-passes-locally-or-on-the-cloud).
> 
>     השירות הוא גרסת בטא, נתון לשינויים.
>     אם יש לך משוב או שאתה רוצה לפנות לצוות הפיתוח, אנא השתמש בערוץ [Qiskit Slack Workspace](https://qiskit.slack.com/archives/C06KF8YHUAU) הזה.

שירות Qiskit Transpiler מספק יכולות טרנספילציה בענן. בנוסף ליכולות Transpiler המקומיות של Qiskit, משימות הטרנספילציה שלך יכולות ליהנות גם ממשאבי ענן של IBM Quantum וגם מפאסי Transpiler מונעי AI.

שירות Qiskit Transpiler מציע ספריית Python לשילוב חלק של שירות זה ויכולותיו בדפוסי ותהליכי עבודה Qiskit הנוכחיים שלך. שירות זה זמין רק למשתמשי IBM Quantum Premium Plan, Flex Plan ו-On-Prem (דרך IBM Quantum Platform API) Plan.

<span id="install-transpiler-service"></span>
## התקנת חבילת qiskit-ibm-transpiler
כדי להשתמש בשירות Qiskit Transpiler, התקן את חבילת `qiskit-ibm-transpiler`:

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="false",
    optimization_level=3,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

החבילה מבצעת אימות אוטומטי באמצעות [אישורי IBM Quantum Platform](/guides/cloud-setup) שלך בהתאם לאופן שבו [Qiskit Runtime מנהל זאת](/guides/initialize-account):
- משתנה סביבה: `QISKIT_IBM_TOKEN`
- קובץ תצורה `~/.qiskit/qiskit-ibm.json` (תחת הסעיף `default-ibm-quantum`).

*הערה*: חבילה זו דורשת Qiskit SDK v1.X.

## אפשרויות טרנספילציה של qiskit-ibm-transpiler
- `backend_name` (אופציונלי, str) - שם Backend כפי שהוא יצפה על ידי QiskitRuntimeService (לדוגמה, `ibm_torino`). אם הוגדר, שיטת ה-transpile משתמשת בפריסה מ-Backend שצוין עבור פעולת הטרנספילציה. אם הוגדרה אפשרות אחרת המשפיעה על הגדרות אלה, כגון `coupling_map`, הגדרות `backend_name` יידחו.
- `coupling_map` (אופציונלי, List[List[int]]) - רשימת coupling map תקינה (לדוגמה, [[0,1],[1,2]]). אם הוגדרה, שיטת ה-transpile משתמשת ב-coupling map זה עבור פעולת הטרנספילציה. אם מוגדר, הוא דוחה כל ערך שצוין עבור `target`.
- `optimization_level` (int) - רמת האופטימיזציה הפוטנציאלית להחיל במהלך תהליך הטרנספילציה. ערכים תקינים הם [1,2,3], כאשר 1 הוא האופטימיזציה המינימלית (והמהירה ביותר), ו-3 האופטימיזציה המרבית (והממושכת ביותר).
- `ai` ("true", "false", "auto") - האם להשתמש ביכולות מונעות AI במהלך הטרנספילציה. יכולות מונעות ה-AI הזמינות יכולות להיות עבור פאסי טרנספילציה של `AIRouting` או שיטות סינתזה מונעות AI אחרות. אם ערך זה הוא `"true"`, השירות מחיל פאסי טרנספילציה מונעות AI שונות בהתאם ל-`optimization_level` המבוקש. אם `"false"`, הוא משתמש בתכונות הטרנספילציה האחרונות של Qiskit ללא AI. לבסוף, אם `"auto"`, השירות מחליט האם להחיל את פאסי ה-heuristic הסטנדרטיים של Qiskit או את הפאסים מונעי ה-AI בהתאם למעגל שלך.
- `qiskit_transpile_options` (dict) - אובייקט מילון Python שיכול לכלול כל אפשרות אחרת תקינה ב[שיטת `transpile()` של Qiskit](defaults-and-configuration-options). אם ה-`qiskit_transpile_options` כולל `optimization_level`, הוא נמחק לטובת ה-`optimization_level` שצוין כפרמטר קלט. אם ה-`qiskit_transpile_options` כולל אפשרות שאינה מוכרת על ידי שיטת `transpile()` של Qiskit, הספרייה תציג שגיאה.

למידע נוסף על שיטות `qiskit-ibm-transpiler` הזמינות, ראה את [תיעוד API של qiskit-ibm-transpiler](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler). למידע נוסף על API השירות, ראה את [תיעוד ה-REST API של שירות Qiskit Transpiler.](https://docs.quantum.ibm.com/api/qiskit-transpiler-service-rest)

## דוגמאות
הדוגמאות הבאות מדגימות כיצד לטרנספל מעגלים באמצעות שירות Qiskit Transpiler עם פרמטרים שונים.

1. צור מעגל וקרא לשירות Qiskit Transpiler לטרנספל את המעגל עם `ibm_torino` כ-`backend_name`, 3 כ-`optimization_level`, וללא שימוש ב-AI במהלך הטרנספילציה.

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="true",
    optimization_level=1,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

*הערה*: תוכל להשתמש רק בהתקנים של backend_name שיש לך גישה אליהם עם חשבון IBM Quantum שלך. מלבד ה-`backend_name`, ה-`TranspilerService` מאפשר גם `coupling_map` כפרמטר.

2. צור מעגל דומה וטרנספל אותו, תוך בקשת יכולות טרנספילציה מונעות AI על ידי הגדרת הדגל `ai` ל-`True`:

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="auto",
    optimization_level=1,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

3. צור מעגל דומה וטרנספל אותו תוך מתן רשות לשירות להחליט האם להשתמש בפאסי טרנספילציה מונעות AI.